# Working with Checkpointables

## Key Concepts

### Checkpoints as a Bundle

An Orbax checkpoint is not just a single, opaque unit. The checkpoint actually consists of a bundle of named objects. In this sense, the checkpoint is really more like a dictionary.

When you save a checkpoint using `ocp.save_pytree`, this produces a bundle with one key/value:

```
{
  'pytree': pytree
}
```

The PyTree is commonly used to represent an ML model state. However, there are other things we want to include in the checkpoint, such as a dataset iterator, which records the exact position in the data where training was interrupted. Now our bundle consists of two keys/values:

```
{
  'pytree': pytree,
  'dataset': dataset,
}
```

### The "Checkpointable" Concept

Each value in this bundle is called a "**checkpointable**", or to use the full term, a "**checkpointable object**".

 This refers to a logical piece of the checkpoint that is distinct in some way from other pieces. There is no hard and fast rule about what constitutes a checkpointable; it is somewhat up to user discretion. Here are a few guidelines, however:

 - Checkpointables are separable.
   - We often wish to load a strict subset of the available checkpointables, because certain checkpointables are not needed for certain cases. For example, in training, we want to load the model state and dataset iterator, but the dataset iterator is not needed for evaluations.
   - Some checkpointables can be excluded entirely from the physical checkpoint without affecting the checkpoint's usefulness. For example, after training completes, the dataset iterator checkpoint is no longer useful, and we can just provide the model parameters to downstream consumers.
 - Checkpointables are often represented by different types, and have different representations on disk. For example, model parameters and optimizer states are represented as large partitioned tensors, while the dataset iterator checkpoint is a simple pointer to a position in the daset iterator.
 - Checkpointables have few, if any, dependencies on other checkpointables within the same checkpoint. The model state can be loaded without the dataset iterator, and vice versa.

### Checkpointable vs. Abstract Checkpointable

A checkpointable is an object that we want to save. Every checkpointable has a mirror image, an **abstract checkpointable** that is used to provide additional information used to guide loading behavior.

For example, consider an array. The array is a concrete object with real data that we want to save. It has properties like shape and dtype. When we load the array from disk, we don't necessarily need any extra information to do so. Orbax can just return the array data with the correct shape and dtype.

However, what if we would like to cast this array (and many others) to a different dtype while loading? This could save some memory, if we cast from e.g. `float64` to `float32`.

We can use an abstract form of the array to specify the properties we need for loading, including `shape` and `dtype`. It will specify `float32` instead of `float64`, which will inform the casting behavior. We can use the shape provided by the abstract array to perform validation, or even to pad / truncate the array during loading if necessary.

Sometimes, loading a checkpoinable never requires additional information. For example, if we just want to load a string dictionary from a JSON file, customization is rarely needed (any necessary customization can be handled via global settings). As a result the abstract checkpointable would be `None` in this case.

To summarize, every checkpointable is a concrete object that we wish to save. Every checkpointable has a corresponding *abstract* checkpointable (which may be None) that stores properties useful for restoring the original checkpointable.

## Concrete Examples

Before beginning, let us note that all APIs demonstrated below have an equivalent function of the same name in `ocp.training.Checkpointer`, for use in the context of a training loop.

Let's take a look at some code examples, starting with some setup.

In [ ]:
### SETUP ###

import numpy as np
from orbax.checkpoint import v1 as ocp
from etils import epath

directory = epath.Path('/tmp/my-checkpoints')

def fake_train_step(step):
  return {
      'params': {
          'layer0': {
              'kernel': np.random.uniform(size=(2, 2)),
              'bias': np.zeros(2),
          }
      },
      'opt_state': {
          '0': np.random.random(size=(2,))
      },
      'step': step,
  }

Previously, we were only saving the train state, which was represeted as a PyTree of arrays.

In [ ]:
train_state = fake_train_step(0)
ocp.save_pytree(directory / 'just_pytree', train_state)

Now, we can save the train state as well as a dataset iterator. We will use [Grain](https://google-grain.readthedocs.io/en/latest/index.html) as an example dataset iterator. TODO(b/409382932): Actually use the Grain iterator to demonstrate this.

In [ ]:
train_state = fake_train_step(1)
ocp.save_checkpointables(
    directory / 'pytree_and_dataset', dict(pytree=train_state, dataset={'a': 1})
)

We can load the checkpointables, which returns a dict with the same keys as are contained in the checkpoint.

In [ ]:
ocp.load_checkpointables(directory / 'pytree_and_dataset')

The APIs `save_pytree` and `load_pytree` interoperate with `save_checkpointables` and `load_checkpointables`. Recall that `save_pytree` just produces a special bundle where the only key is "pytree" and the only checkpointable is the PyTree object. Calling `load_pytree` will only load the key named `pytree`, regardless of what other checkpointables are present.

In [ ]:
ocp.load_pytree(directory / 'pytree_and_dataset')

Note that we can give any names we want to our checkpointables.

In [ ]:
ocp.save_checkpointables(
    directory / 'train_state_and_dataset', dict(train_state=train_state, dataset={'a': 1})
)

These are still loadable as normal with `load_checkpointables`, but `load_pytree` will fail because there is no key named "pytree".

In [ ]:
ocp.load_checkpointables(directory / 'train_state_and_dataset')

In [ ]:
try:
  ocp.load_pytree(directory / 'train_state_and_dataset')
except BaseException as e:
  print(e)

As mentioned above, a key property of checkpointables is that they are easily separable. We should be able to restore only `train_state` or only `dataset` individually. We need to call `load_checkpointables` and specify that only the `dataset` key is desired.

In [ ]:
ocp.load_checkpointables(
    directory / 'train_state_and_dataset',
    dict(dataset=None),
)

Alternatively, we can use the same API to load the train state alone.

In [ ]:
ocp.load_checkpointables(
    directory / 'train_state_and_dataset',
    dict(train_state=None),
)

What is the meaning of the syntax `dict(train_state=None)` or `dict(dataset=None)`?

We must recall the concept of abstract checkpointables from above. Just as `save_checkpointables` allows specifying a dictionary of checkpointables, `load_checkpointables` allows specifying a dictionary of *abstract* checkpointables.

In the example above, we specify `None` as the abstract checkpointable for `train_state`. `None` is always a valid value for an abstract checkpointable. It is equivalent to saying:

> *Restore this checkpointable exactly as it was saved - you don't need any additional information from me.*

So the pattern:

```
ocp.laod_checkpointables(path, dict(checkpointable_name=None))
```

is an effective way of loading while filtering out only the keys that are desired.